# EDW REST client (`georest.esri.edw`) - usage examples

Walks through every public function in `georest.esri.edw` against the live USFS Enterprise
Data Warehouse (EDW) ArcGIS REST services:

- `search_edw_services` - find services by keyword or theme
- `get_service_info` - service-level metadata (layers, spatial reference)
- `get_layer_info` - layer-level metadata (fields, capabilities)
- `get_layer_metadata` - FGDC/ISO metadata (abstract, field definitions, domains)
- `query_features` - attribute and spatial feature queries
- `query_features_with_pagination` - queries beyond the 2,000-record server cap
- `query_features_analytic` - SQL window functions (RANK, ROW_NUMBER, ...)
- `top_n_per_group` - a convenience wrapper for "top N per group" queries

The edw functions return GeoJSON and plain records, because features carry real geometry
(polygons, lines) and `georest.esri` uses only the standard library. `georest.to_frame` turns
any of them into a pandas DataFrame, which is how results are shown here.

## Setup

```
pip install georest
```

or, from a clone of this repository, `pip install -e .` from the repo root. The map uses
`folium` (`pip install "georest[examples]"`).

In [1]:
import pandas as pd

import georest
from georest.esri.edw import (
    search_edw_services,
    get_service_info,
    get_layer_info,
    get_layer_metadata,
    query_features,
    query_features_with_pagination,
    query_features_analytic,
    top_n_per_group,
)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 100)

## Find services by keyword or theme: `search_edw_services`

Matches on service name, theme description, and a keyword-alias table (for example "riparian"
pulls in inland-waters and hydro services even though the word never appears in a service
name).

In [2]:
# A plain keyword search
georest.to_frame(search_edw_services("fire"))[["name", "theme", "description"]]

,name,theme,description
0,EDW_AerialFireRetardantAvoidanceAreas_Aquatic_01,inland_waters,This data depicts aquatic aerial fire retardant avoidance areas delivered as part of the 2011 Nationwide Aerial Application of Fire Retardant on…
1,EDW_AerialFireRetardantAvoidanceAreas_Terrestrial_01,environment,This data depicts terrestrial aerial fire retardant avoidance areas delivered as part of the 2011 Nationwide Aerial Application of Fire Retardant on…
2,EDW_BurnedAreaEmergencyResponse_01,environment,"An initial Burned Area Reflectance Classification (BARC) dataset is created by analyzing a differenced Normalized Burn Ratio (dNBR) image, which was…"
3,EDW_CommWildfireDefenseGrant_01,environment,Community Wildfire Defense Grants (CWDG) is a program administered by the Forest Service to help at-risk communities and Tribes plan for and reduce…
4,EDW_FireOccurrenceAndPerimeter_01,environment,The FirePerimeterFinal polygon layer represents final mapped wildland fire perimeters.
5,EDW_FireOccurrenceCurrentEdition_01,environment,EDW_FireOccurrenceCurrentEdition_01 provides the current edition of Karen C. Short’s Spatial wildfire occurrence data for the United States (FPA FOD).…
6,EDW_FireOccurrenceFIRESTAT_YRLY_01,environment,"The FIRESTAT (Fire Statistics System) Fire Occurrence point layer represents ignition points, or points of origin, from which individual wildland…"
7,EDW_FireshedRegistry_01,environment,The Fireshed Registry is a geospatial dashboard and decision tool built to organize information about wildfire transmission to communities and…
8,EDW_ForeststoFaucets_02,inland_waters,"Forests to Faucets 2.0 builds upon the national Forests to Faucets(2011) by updating base data and adding new threats including wildfire, invasive…"
9,EDW_MTBS_01,environment,"The Monitoring Trends in Burn Severity (MTBS) Program assesses the frequency, extent, and magnitude (size and severity) of all large wildland fires…"


In [3]:
# Keyword-alias expansion: "riparian" is in no service name, but resolves to inland_waters,
# hydro, watershed and aquatic services
georest.to_frame(search_edw_services("riparian"))[["name", "theme"]]

,name,theme
0,EDW_AerialFireRetardantAvoidanceAreas_Aquatic_01,inland_waters
1,EDW_AquaticOrganismPassage_01,environment
2,EDW_ExperimentalForestandRange_01,boundaries
3,EDW_ForeststoFaucets_02,inland_waters
4,EDW_GreatBasinMountainRangesWatersheds_01,inland_waters
5,EDW_HydroFlowMetrics2040_01,environment
6,EDW_HydroFlowMetrics2080_01,environment
7,EDW_HydroFlowMetricsAbsChange2040_01,environment
8,EDW_HydroFlowMetricsAbsChange2080_01,environment
9,EDW_HydroFlowMetricsHistorical_01,environment


In [4]:
# Filter by theme alone: every transportation service
georest.to_frame(search_edw_services(theme="transportation"))[["name", "type", "description"]]

,name,type,description
0,EDW_MVUM_01,MapServer,The feature class indicates the specific types of motorized vehicles allowed on the designated routes and their seasons of use.
1,EDW_MVUM_02,MapServer,A map service on the www depicting Forest Service roads and trails that are designated for motor vehicle use under the official U.S.
2,EDW_RoadBasic_01,MapServer,Existing Forest Service roads with attributes representing their characteristics.
3,EDW_TrailNFSPublish_01,MapServer,The TrailNFS_Publish Layer is designed to provide information about National Forest System trail locations and characteristics to the public.
4,EDW_TrailNFSPublishWithDataStatus_01,MapServer,A map service on the world wide web that depicts National Forest Service trails that have been approved for publication.


## Service-level metadata: `get_service_info`

Returns the description, spatial reference, extent, and the list of layers in a MapServer.

In [5]:
service_name = "EDW_MTBS_01"   # Monitoring Trends in Burn Severity
info = get_service_info(service_name)
print(info["description"])
print()
print(len(info["layers"]), "layers")
georest.to_frame(info["layers"])

A map service on the www that depicts Fire Occurrence Locations and Burned Area Boundaries from the beginning of the Landsat Thematic Mapper archive to the present. The Monitoring Trends in Burn Severity (MTBS) project maps the location, extent, and severity of all large fires in the conterminous United States (CONUS), Alaska, Hawaii and Puerto Rico from the beginning of the Landsat Thematic Mapper archive to the present. All documented fires greater than 1,000 acres in the western U.S. and greater than 500 acres in the eastern U.S. are mapped across all ownerships. The project produces geospatial and tabular data for analysis at a range of spatial, temporal, and thematic scales and are intended to meet a variety of information needs that require consistent data about fire effects through space and time. MTBS is conducted through a partnership between the U.S. Geological Survey National Center for Earth Resources Observation and Science (EROS) and the USDA Forest Service. Metadata: htt

,id,name,defaultVisibility,minScale,maxScale
0,87,2026 Fire Occurrence Locations,True,0,0
1,86,2026 Burned Area Boundaries,True,0,0
2,85,2025 Fire Occurrence Locations,False,0,0
3,84,2025 Burned Area Boundaries,False,0,0
4,83,2024 Fire Occurrence Locations,False,0,0
5,82,2024 Burned Area Boundaries,False,0,0
6,81,2023 Fire Occurrence Locations,False,0,0
7,80,2023 Burned Area Boundaries,False,0,0
8,78,2022 Fire Occurrence Locations,False,0,0
9,79,2022 Burned Area Boundaries,False,0,0


## Layer-level metadata: `get_layer_info`

Fields, geometry type, and which operations and analytics the layer supports
(`capabilities`, `advancedQueryCapabilities`).

In [6]:
layer_id = 63  # "Burned Area Boundaries (All Years)"
layer_info = get_layer_info(service_name, layer_id)
print(layer_info["name"], "-", layer_info["geometryType"])
print("capabilities:", layer_info["capabilities"])
georest.to_frame(layer_info["fields"])

Burned Area Boundaries (All Years) - esriGeometryPolygon
capabilities: ['Map', 'Query', 'Data']


,name,type,alias
0,objectid,OID,OBJECTID
1,fire_id,String,FIRE_ID
2,fire_name,String,FIRE_NAME
3,year,SmallInteger,YEAR
4,startmonth,SmallInteger,STARTMONTH
5,startday,SmallInteger,STARTDAY
6,fire_type,String,FIRE_TYPE
7,acres,Double,ACRES
8,irwinid,String,IRWINID
9,map_id,Integer,MAP_ID


## FGDC/ISO metadata: `get_layer_metadata`

A separate document from `get_layer_info`: it carries the dataset abstract and purpose and
per-field *definitions* (what a field actually means), plus optional coded-value and range
domains.

In [7]:
meta = get_layer_metadata(service_name, layer_id, include_domains=True)
print(meta["title"])
print()
print(meta["abstract"])
print()
print("keywords:", ", ".join(meta["keywords"]))

MTBS_Burn_Area_Boundary

The Monitoring Trends in Burn Severity (MTBS) Program assesses the frequency, extent, and magnitude (size and severity) of all large wildland fires (including wildfires and prescribed fires) in the conterminous United States (CONUS), Alaska, Hawaii, and Puerto Rico from the beginning of the Landsat Thematic Mapper archive to the present. All fires reported as greater than 1,000 acres in the western U.S. and greater than 500 acres in the eastern U.S. are mapped across all ownerships. MTBS produces a series of geospatial and tabular data for analysis at a range of spatial, temporal, and thematic scales and are intended to meet a variety of information needs that require consistent data about fire effects through space and time. This map layer is a vector polygon shapefile of the location of all currently inventoried fires occurring between calendar year 1984 and the current MTBS release for CONUS, Alaska, Hawaii and Puerto Rico. Please visit https://mtbs.gov/anno

In [8]:
# Field definitions (only some fields are documented by the data provider)
fields = georest.to_frame(meta["attributes"])
fields["definition"] = fields["definition"].fillna("(undocumented)")
fields[["name", "definition"]]

,name,definition
0,OBJECTID,Internal feature number.
1,FIRE_ID,
2,FIRE_NAME,
3,YEAR,
4,STARTMONTH,
5,STARTDAY,
6,FIRE_TYPE,
7,ACRES,
8,Post_ID,Landsat or Sentinel post scene ID.
9,Pre_ID,Landsat or Sentinel pre scene ID.


## Attribute and spatial queries: `query_features`

Returns a GeoJSON `FeatureCollection`. `where` filters by attributes; `geometry`,
`geometry_type` and `spatial_rel` add a spatial filter. `georest.to_frame` gives the attribute
table; the FeatureCollection keeps the shapes, for a map.

In [9]:
# An attribute-only query: one named fire
cameron_peak = query_features(service_name, layer_id,
                              where="fire_name = 'CAMERON PEAK'",
                              out_fields="fire_name,ig_date,acres")
georest.to_frame(cameron_peak)

,id,fire_name,ig_date,acres
0,0,CAMERON PEAK,20200813,208743


In [10]:
# return_count_only skips geometry and attributes when only a number is needed
query_features(service_name, layer_id, where="acres > 100000", return_count_only=True)["count"]

308

In [11]:
# A spatial query, using a forest boundary as the area of interest.
# The Idaho Panhandle NF boundary is a 161-part multipolygon with holes; query_features falls
# back to an ID-based fetch for an area this complex (see its docstring), so this just works.
ipnf = query_features("EDW_ForestSystemBoundaries_01", 0, where="FORESTORGCODE='0104'")
aoi_geom = ipnf["features"][0]["geometry"]

# ...then the fires intersecting it. The geometry can be GeoJSON, Esri JSON, or a bbox string.
ipnf_fires = query_features(service_name, layer_id,
                            geometry=aoi_geom, geometry_type="esriGeometryPolygon",
                            out_fields="fire_name,year,acres")
fires_df = georest.to_frame(ipnf_fires).sort_values("year", ascending=False)
print(len(fires_df), "fires intersect the Idaho Panhandle National Forests")
fires_df

40 fires intersect the Idaho Panhandle National Forests


,id,fire_name,year,acres
39,39,RIDGE CREEK,2023,4600
38,38,COLUMBUS,2022,2573
35,35,CALEDONIA,2022,1451
37,37,THOR,2022,2097
36,36,DIAMOND WATCH,2022,1432
34,34,RUSSELL MOUNTAIN,2022,26104
27,27,DECEITFUL,2021,12748
30,30,TUMBLEDOWN,2021,11258
31,31,DEVILS PULPIT,2021,3185
32,32,STATELINE COMPLEX,2021,1958


For a map, full-detail shapes are far more than the eye needs: the forest boundary alone is
megabytes. `max_allowable_offset` asks the server to simplify each shape so no vertex moves
more than the given distance (0.002 degrees is about 200 m). The query above kept full
detail, so which fires intersect the forest is exact; only these copies, drawn below, are
simplified.

In [12]:
import folium

ipnf_map = query_features("EDW_ForestSystemBoundaries_01", 0, where="FORESTORGCODE='0104'",
                          max_allowable_offset=0.002)
fires_map = query_features(service_name, layer_id, geometry=aoi_geom,
                           geometry_type="esriGeometryPolygon",
                           out_fields="fire_name,year,acres", max_allowable_offset=0.002)

m = folium.Map(location=[47.6, -116.3], zoom_start=7,
               tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer/tile/{z}/{y}/{x}",
               attr="Esri World Topo")
folium.GeoJson(ipnf_map, name="Idaho Panhandle NF",
               style_function=lambda f: {"color": "#2e6b30", "weight": 1, "fillOpacity": 0.08}).add_to(m)
folium.GeoJson(fires_map, name="MTBS fires",
               style_function=lambda f: {"color": "#c0392b", "weight": 1, "fillOpacity": 0.45},
               tooltip=folium.GeoJsonTooltip(fields=["fire_name", "year", "acres"])).add_to(m)
folium.LayerControl().add_to(m)
m

In [13]:
# A plain bbox string also works as the geometry filter (west, south, east, north)
bbox_fires = query_features(service_name, layer_id,
                            geometry="-116.5,47.5,-116.0,48.0",
                            geometry_type="esriGeometryEnvelope",
                            out_fields="fire_name,year,acres", max_features=5)
georest.to_frame(bbox_fires)

,id,fire_name,year,acres
0,0,ULM PEAK,2006,4606
1,1,CAPE HORN,2015,1505
2,2,NORTH GRIZZLY,2015,5284
3,3,WHITETAIL,2015,2022
4,4,LOWER FLAT,2015,9023


## Beyond the 2,000-record server cap: `query_features_with_pagination`

The same signature as `query_features`, but `max_features` can exceed the server's 2,000-record
page; it pages through `resultOffset` automatically. This uses the point layer (fire ignition
locations) rather than the polygon layer, because the EDW server answers 500 on
exactly-2,000-row pages of heavy polygon geometry, a server-side quirk unrelated to this
function.

In [14]:
all_fires = query_features_with_pagination(service_name, 62,  # "Fire Occurrence Locations (All Years)"
                                           out_fields="fire_name,ig_date,acres",
                                           max_features=4000)
points = georest.to_frame(all_fires)
print(len(points), "fire locations, with lon and lat")
points.describe()[["acres", "lon", "lat"]]

4000 fire locations, with lon and lat


,acres,lon,lat
count,4000.000000,4000.000000,4000.000000
mean,6715.486000,-100.092435,36.529675
std,20581.634348,16.112381,7.427888
min,500.000000,-164.330882,18.144036
25%,1169.000000,-114.181571,30.579696
50%,1960.000000,-100.254525,35.395114
75%,4624.250000,-84.259112,42.031657
max,457680.000000,-65.350627,70.139256


## SQL window functions: `query_features_analytic`

Runs ArcGIS's `queryAnalytic` operation. Unlike `query_features`, rows are not collapsed: each
analytic value is appended as a new field on its source feature.

Filter on the computed field by the `out_name` you gave it, in `analytic_where`. A ranking
function such as `RANK` takes no `field`: the field being ranked is named in `order_by`.

In [15]:
# Rank fires by acreage within each ignition year; keep the largest fire of each year
biggest_per_year = query_features_analytic(
    service_name, layer_id,
    out_analytics=[{"type": "RANK", "order_by": "acres DESC", "out_name": "acres_rank"}],
    partition_by="year",
    analytic_where="acres_rank = 1",
    out_fields="fire_name,year,acres",
    return_geometry=False,
)
georest.to_frame(biggest_per_year).sort_values("year", ascending=False)

,id,fire_name,year,acres,acres_rank
42,42,MORRILL,2026,645316.0,1
41,41,COTTONWOOD PEAK,2025,137796.0,1
40,40,SMOKEHOUSE CREEK,2024,1047245.0,1
39,39,YORK,2023,94728.0,1
38,38,HERMITS PEAK,2022,351782.0,1
37,37,DIXIE,2021,979722.0,1
36,36,AUGUST COMPLEX,2020,1067970.0,1
35,35,CHALKYITSIK COMPLEX,2019,470271.0,1
34,34,MARTIN,2018,431841.0,1
33,33,OKS - STARBUCK,2017,656868.0,1


**Other analytic types:** the Esri `queryAnalytic` spec also defines `SUM`, `AVG`, `MIN`,
`MAX`, `COUNT`, `STDDEV`, `VAR`, `LAG`, `LEAD`, `NTILE`, `FIRST_VALUE`, `LAST_VALUE`,
`PERCENTILE_CONT` and `PERCENTILE_DISC`, but this EDW server only supports the ranking family
in practice: `RANK` (above), `DENSE_RANK`, `ROW_NUMBER`, `PERCENT_RANK` and `CUME_DIST`. The
rest fail with `"Unable to complete operation"` (or an explicit `"not supported"` for the
percentile functions), however they are called.

`ROW_NUMBER` is a useful contrast to `RANK`: `RANK` gives tied values the *same* rank, so a tie
for the smallest or largest fire in a year produces several rows for that year (2024 has two
fires tied at exactly 501 acres). `ROW_NUMBER` breaks ties arbitrarily but deterministically,
guaranteeing exactly one row per partition.

In [16]:
# The smallest fire per year via ROW_NUMBER: exactly one row per year, even in years
# (like 2024) where two fires tie at exactly the same acreage
smallest_per_year = query_features_analytic(
    service_name, layer_id,
    out_analytics=[{"type": "ROW_NUMBER", "order_by": "acres ASC", "out_name": "row_num"}],
    partition_by="year",
    analytic_where="row_num = 1",
    out_fields="fire_name,year,acres",
    return_geometry=False,
)
smallest = georest.to_frame(smallest_per_year).sort_values("year", ascending=False)
print(len(smallest), "rows, one per year")
smallest

43 rows, one per year


,id,fire_name,year,acres,row_num
42,42,SALINA CANYON EAST RX,2026,5058.0,1
41,41,CAHAS MOUNTTAIN FIRE,2025,501.0,1
40,40,CON RKC 01 RX,2024,501.0,1
39,39,ROUND MOUNTAIN,2023,500.0,1
38,38,C-136 QUAIL 26/32-137B QUAIL 28 FM,2022,502.0,1
37,37,CALVIN LEE RD,2021,497.0,1
36,36,CHICKASAWHAY BB 12,2020,500.0,1
35,35,RX - LAKE SHADLEY,2019,500.0,1
34,34,UNNAMED,2018,518.0,1
33,33,RIVER OTTER,2017,502.0,1


## A convenience wrapper: `top_n_per_group`

The same query as above, without hand-rolling the RANK analytic. `descending=False` ranks
smallest first.

In [17]:
biggest_by_year = top_n_per_group(service_name, layer_id, field="acres", group_by="year",
                                  descending=True, n=1, out_fields="fire_name,year,acres")
georest.to_frame(biggest_by_year).sort_values("year", ascending=False)

,id,fire_name,year,acres,rank_val
42,42,MORRILL,2026,645316.0,1
41,41,COTTONWOOD PEAK,2025,137796.0,1
40,40,SMOKEHOUSE CREEK,2024,1047245.0,1
39,39,YORK,2023,94728.0,1
38,38,HERMITS PEAK,2022,351782.0,1
37,37,DIXIE,2021,979722.0,1
36,36,AUGUST COMPLEX,2020,1067970.0,1
35,35,CHALKYITSIK COMPLEX,2019,470271.0,1
34,34,MARTIN,2018,431841.0,1
33,33,OKS - STARBUCK,2017,656868.0,1


The smallest fire by year. Note that this function returns more than N rows for a group when
there is a tie (see the two 2024 rows below). For exactly one row per group, use
`query_features_analytic()` with `ROW_NUMBER`, as above.

In [18]:
smallest_by_year = top_n_per_group(service_name, layer_id, field="acres", group_by="year",
                                   descending=False, n=1, out_fields="fire_name,year,acres")
georest.to_frame(smallest_by_year).sort_values("year", ascending=False)

,id,fire_name,year,acres,rank_val
50,50,SALINA CANYON EAST RX,2026,5058.0,1
49,49,CAHAS MOUNTTAIN FIRE,2025,501.0,1
47,47,C-212 BB1 LC RX,2024,501.0,1
48,48,CON RKC 01 RX,2024,501.0,1
45,45,BOGGY HOLLOW SOUTH,2023,500.0,1
46,46,ROUND MOUNTAIN,2023,500.0,1
44,44,C-136 QUAIL 26/32-137B QUAIL 28 FM,2022,502.0,1
43,43,CALVIN LEE RD,2021,497.0,1
42,42,CHICKASAWHAY BB 12,2020,500.0,1
40,40,UNNAMED,2019,500.0,1


The largest fire that started in each month, across all years.

In [19]:
biggest_by_month = top_n_per_group(service_name, layer_id, field="acres", group_by="startmonth",
                                   descending=True, n=1, out_fields="fire_name,year,startmonth,acres")
georest.to_frame(biggest_by_month).sort_values("startmonth")

,id,fire_name,year,startmonth,acres,rank_val
0,0,UNNAMED,2014,1,60996.0,1
1,1,SMOKEHOUSE CREEK,2024,2,1047245.0,1
2,2,OKS - STARBUCK,2017,3,656868.0,1
3,3,HERMITS PEAK,2022,4,351782.0,1
4,4,WALLOW,2011,5,563564.0,1
5,5,BOUNDARY,2004,6,537728.0,1
6,6,DIXIE,2021,7,979722.0,1
7,7,AUGUST COMPLEX,2020,8,1067970.0,1
8,8,CREEK,2020,9,381529.0,1
9,9,CEDAR,2003,10,268142.0,1


The 10 largest fires of 2017.

In [20]:
largest_2017 = top_n_per_group(service_name, layer_id, field="acres", group_by="year",
                               descending=True, n=10, where="year=2017",
                               out_fields="fire_name,year,startmonth,acres")
georest.to_frame(largest_2017).sort_values("rank_val")

,id,fire_name,year,startmonth,acres,rank_val
0,0,OKS - STARBUCK,2017,3,656868.0,1
1,1,PERRYTON,2017,3,290085.0,2
2,2,THOMAS,2017,12,282076.0,3
3,3,BRIDGE COULEE,2017,7,222645.0,4
4,4,ROOSTERS COMB,2017,7,217340.0,5
5,5,CHETCO BAR,2017,7,194761.0,6
6,6,RICE RIDGE,2017,7,171675.0,7
7,7,WEST MIMS,2017,4,166678.0,8
8,8,CAMPBELL RIVER,2017,6,161363.0,9
9,9,SNOWSTORM,2017,7,153432.0,10
